<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 35 · Deep Q-Network</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Thay bảng Q bằng mạng nơ-ron, cài DQN với replay buffer và mạng đích, huấn luyện trên CartPole tới 500 điểm, rồi bỏ từng mẹo để thấy vì sao cần nó. Mỗi lần huấn luyện khoảng 40 giây trên CPU, không cần GPU.</p>
</div>

### Sau notebook này bạn sẽ

- Hiểu vì sao Q-learning với mạng nơ-ron dễ phân kỳ (tam giác chết).
- Cài replay buffer, mạng đích, epsilon giảm dần và mục tiêu tách terminated với truncated.
- Huấn luyện DQN trên CartPole, giữ bản tốt nhất, đo lại trên tập mới.
- Bỏ mạng đích, bỏ replay, thử Double DQN, và đọc kết quả qua nhiều seed.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 5 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/01-tu-bang-sang-mang/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Từ bảng sang mạng](#trang-1)
2. [Trang 2 · Replay buffer và mạng đích](#trang-2)
3. [Trang 3 · Huấn luyện trên CartPole](#trang-3)
4. [Trang 4 · Bỏ từng mẹo](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Từ bảng sang mạng

CartPole, mốc chọn bừa, và mạng Q.

*Đọc kèm trên web: [Trang 1 · Từ bảng sang mạng](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/01-tu-bang-sang-mang/)*

In [ ]:
!pip install -q gymnasium                                   # thư viện môi trường RL, Colab chưa chắc có sẵn

In [ ]:
import copy                                                 # sao chép mạng
import gymnasium as gym                                     # môi trường RL
import numpy as np                                          # mảng
import torch                                                # PyTorch
import torch.nn as nn                                       # các lớp mạng
import torch.nn.functional as F                             # hàm loss
torch.set_num_threads(1)                                    # mạng nhỏ: một luồng nhanh hơn và lặp lại được
env = gym.make("CartPole-v1")                               # xe đẩy và cây sào
print(env.observation_space.shape, env.action_space, env.spec.max_episode_steps)   # 4 số, 2 hành động, tối đa 500 bước
s, _ = env.reset(seed=0)                                    # trạng thái đầu
print(s)                                                    # vị trí, vận tốc, góc, vận tốc góc

Output mong đợi:

```text
(4,) Discrete(2) 500
[ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
```

In [ ]:
rng = np.random.default_rng(0)                              # ngẫu nhiên của tác tử
env.reset(seed=0)                                           # seed môi trường
diem = []                                                   # điểm mỗi tập
for _ in range(100):                                        # 100 tập
    env.reset()                                             # tập mới
    G = 0                                                   # điểm tập này
    while True:                                             # mỗi bước
        _, r, ket_thuc, cat_ngang, _ = env.step(int(rng.integers(2)))   # đẩy bừa trái hoặc phải
        G += r                                              # cộng thưởng
        if ket_thuc or cat_ngang:                           # sào đổ hoặc đủ 500 bước
            break                                           # hết tập
    diem.append(G)                                          # ghi lại
print(np.mean(diem), np.max(diem))                          # trung bình, tốt nhất

Output mong đợi:

```text
21.77 72.0
```

In [ ]:
def tao_mang():                                             # MLP: 4 số vào, Q của 2 hành động ra
    return nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))   # hai lớp ẩn 128
torch.manual_seed(0)                                        # khởi tạo cố định
q = tao_mang()                                              # mạng chưa học
print(sum(p.numel() for p in q.parameters()), q(torch.as_tensor(s)).detach().numpy().round(3))   # số tham số, Q ở trạng thái đầu

Output mong đợi:

```text
17410 [ 0.013 -0.048]
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Replay buffer và mạng đích

Cài DQN: replay buffer, mạng đích, epsilon giảm dần, đánh giá định kỳ.

*Đọc kèm trên web: [Trang 2 · Replay buffer và mạng đích](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/02-replay-va-mang-dich/)*

In [ ]:
def danh_gia(mang, n=20, seed=10_000):                      # điểm trung bình của chính sách tham lam
    e = gym.make("CartPole-v1")                             # môi trường riêng để đánh giá
    tong = []                                               # điểm mỗi tập
    for i in range(n):                                      # n tập với seed cố định
        s, _ = e.reset(seed=seed + i)                       # tập thứ i
        G = 0                                               # điểm
        while True:                                         # mỗi bước
            with torch.no_grad():                           # không cần gradient
                a = int(mang(torch.as_tensor(s)).argmax())  # tham lam, không khám phá
            s, r, ket_thuc, cat_ngang, _ = e.step(a)        # đi
            G += r                                          # cộng thưởng
            if ket_thuc or cat_ngang:                       # hết tập
                break                                       # ra khỏi vòng
        tong.append(G)                                      # ghi lại
    return float(np.mean(tong))                             # trung bình
def dqn(n_buoc=50_000, seed=0, muc_tieu_rieng=True, replay=True, double=False, ket_thuc_khi_cat=False,
        lr=5e-4, cap_nhat_dich=100, bo_nho=50_000, lo=128, gamma=0.99, eps_cuoi=0.05, eps_buoc=10_000,
        bat_dau=1000, danh_gia_moi=5000):                   # DQN; các cờ dùng cho thí nghiệm trang 4 và bài tập
    torch.manual_seed(seed)                                 # khởi tạo mạng
    rng = np.random.default_rng(seed)                       # khám phá và rút lô
    q = tao_mang()                                          # mạng chính
    q_dich = copy.deepcopy(q)                               # mạng đích: bản sao đóng băng
    opt = torch.optim.Adam(q.parameters(), lr=lr, fused=True)   # fused: lặp lại được trên CPU
    S, S2 = np.zeros((bo_nho, 4), np.float32), np.zeros((bo_nho, 4), np.float32)   # replay buffer: trạng thái, trạng thái mới
    A, Rw, D = np.zeros(bo_nho, np.int64), np.zeros(bo_nho, np.float32), np.zeros(bo_nho, np.float32)   # hành động, thưởng, cờ kết thúc
    n, G, diem_tap = 0, 0, []                               # số bước đã lưu, điểm tập hiện tại, điểm các tập
    tot_nhat, ban_tot = -1, None                            # bản tốt nhất theo đánh giá
    s, _ = env.reset(seed=seed)                             # tập đầu
    for t in range(1, n_buoc + 1):                          # mỗi bước môi trường
        eps = max(eps_cuoi, 1 - (1 - eps_cuoi) * t / eps_buoc)   # giảm tuyến tính rồi giữ nguyên
        if rng.random() < eps:                              # khám phá
            a = int(rng.integers(2))                        # hành động ngẫu nhiên
        else:                                               # khai thác
            with torch.no_grad():                           # không cần gradient
                a = int(q(torch.as_tensor(s)).argmax())     # hành động tham lam
        s2, r, ket_thuc, cat_ngang, _ = env.step(a)         # đi một bước
        G += r                                              # cộng thưởng
        i = n % bo_nho                                      # ghi đè vòng tròn
        S[i], A[i], Rw[i], S2[i] = s, a, r, s2              # lưu bước
        D[i] = float(ket_thuc or (ket_thuc_khi_cat and cat_ngang))   # chỉ kết thúc thật mới bỏ phần tương lai
        n += 1                                              # đếm
        s = s2                                              # sang trạng thái mới
        if ket_thuc or cat_ngang:                           # hết tập
            diem_tap.append(G)                              # ghi điểm
            G = 0                                           # tập mới
            s, _ = env.reset()                              # về đầu
        if n >= bat_dau:                                    # đủ dữ liệu thì học
            idx = rng.integers(min(n, bo_nho), size=lo) if replay else np.array([i])   # lô ngẫu nhiên, hoặc chỉ bước vừa đi
            sb, ab, rb, s2b, db = map(torch.as_tensor, (S[idx], A[idx], Rw[idx], S2[idx], D[idx]))   # sang tensor
            with torch.no_grad():                           # mục tiêu không nhận gradient
                mang_dich = q_dich if muc_tieu_rieng else q # có hay không có mạng đích
                if double:                                  # Double DQN
                    q_sau = mang_dich(s2b).gather(1, q(s2b).argmax(1, keepdim=True)).squeeze(1)   # mạng chính chọn, mạng đích chấm
                else:                                       # DQN chuẩn
                    q_sau = mang_dich(s2b).max(1).values    # max theo mạng đích
                y = rb + gamma * (1 - db) * q_sau           # mục tiêu
            q_sa = q(sb).gather(1, ab.unsqueeze(1)).squeeze(1)   # Q của đúng hành động đã làm
            loss = F.smooth_l1_loss(q_sa, y)                # Huber
            opt.zero_grad()                                 # xóa gradient
            loss.backward()                                 # lượt ngược
            opt.step()                                      # cập nhật
            if t % cap_nhat_dich == 0:                      # định kỳ
                q_dich.load_state_dict(q.state_dict())      # chép sang mạng đích
        if t % danh_gia_moi == 0:                           # đánh giá định kỳ
            diem = danh_gia(q)                              # chính sách tham lam, 20 tập
            print(t, len(diem_tap), round(np.mean(diem_tap[-20:]), 1), diem)   # bước, số tập, điểm lúc học, điểm đánh giá
            if diem > tot_nhat:                             # tốt hơn mọi lần trước
                tot_nhat, ban_tot = diem, copy.deepcopy(q)  # giữ bản này
    return ban_tot, q                                       # bản tốt nhất và bản cuối
print(danh_gia(q))                                          # mạng chưa học của trang 1

Output mong đợi:

```text
91.05
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Mạng chưa học đã giữ sào 91 bước, gấp bốn lần chọn bừa: argmax của một mạng ngẫu nhiên vẫn đổi hướng theo góc sào, tình cờ thành một phản xạ thô.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Huấn luyện trên CartPole

Một lần huấn luyện (khoảng 40 giây), rồi đo lại trên 100 tập mới.

*Đọc kèm trên web: [Trang 3 · Huấn luyện trên CartPole](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/03-huan-luyen-cartpole/)*

In [ ]:
ban_tot, ban_cuoi = dqn()                                   # seed 0, 50 000 bước

Output mong đợi:

```text
5000 132 62.0 177.35
10000 193 72.6 95.2
15000 245 98.0 101.2
20000 296 100.7 129.95
25000 349 82.8 90.05
30000 369 252.4 151.1
35000 421 78.2 104.2
40000 458 164.1 500.0
45000 537 32.6 56.15
50000 564 220.6 500.0
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cột cuối là điểm của chính sách tham lam: dao động 90 tới 177, chạm 500 ở bước 40 000, rơi xuống 56 ở bước 45 000, rồi lại 500. Đường học của DQN không đi lên đều.</p>
</div>

In [ ]:
print(danh_gia(ban_tot, n=100, seed=20_000), danh_gia(ban_cuoi, n=100, seed=20_000))   # 100 tập chưa dùng lần nào

Output mong đợi:

```text
500.0 500.0
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Bỏ từng mẹo

Bỏ mạng đích, bỏ replay buffer.

*Đọc kèm trên web: [Trang 4 · Bỏ từng mẹo](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/04-bo-tung-meo/)*

In [ ]:
_, q_khong_dich = dqn(muc_tieu_rieng=False)                 # mục tiêu tính bằng chính mạng đang học
with torch.no_grad():                                       # không cần gradient
    print(q_khong_dich(torch.as_tensor(s)).numpy())         # Q ở trạng thái đầu

Output mong đợi:

```text
5000 254 16.9 9.45
10000 691 9.1 9.45
15000 1212 9.3 9.45
20000 1733 9.5 9.45
25000 2250 9.7 9.45
30000 2771 9.3 9.45
35000 3289 9.8 9.45
40000 3809 9.6 9.45
45000 4332 9.4 9.45
50000 4849 9.8 9.45
[1.9254470e+08 2.6113987e+08]
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">PHÂN KỲ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Q lên hàng trăm triệu trong khi giá trị thật không thể vượt 100: phân kỳ. Chính sách luôn đẩy về một phía và sào đổ sau khoảng 9 bước.</p>
</div>

In [ ]:
_ = dqn(replay=False)                                       # chỉ học từ bước vừa đi, vẫn có mạng đích

Output mong đợi:

```text
5000 193 29.4 19.3
10000 450 10.2 14.8
15000 729 56.0 45.7
20000 1034 31.0 14.1
25000 1315 13.8 66.2
30000 1559 51.6 49.45
35000 1678 56.8 60.7
40000 1861 28.0 93.5
45000 2175 9.8 9.45
50000 2360 13.0 11.9
```

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án. Mỗi bài khoảng 40 giây.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch35-dqn/05-bai-tap/)*

### Bài 35.1 · Đánh giá quá cao, đo bằng số

So max Q ở trạng thái đầu với lợi nhuận chiết khấu thật 99,34, cho DQN và Double DQN.
<details><summary>Xem đáp án</summary>

DQN dự đoán 134,77, cao hơn cả trần 100. Double DQN 127,63 ở seed này, nhưng trên 5 seed trung bình 137,0 so với 128,1: trên CartPole với mạng đích chép mỗi 100 bước, Double DQN không giúp. Code ở ô ngay dưới.
</details>

In [ ]:
that = sum(0.99 ** k for k in range(500))                   # lợi nhuận chiết khấu thật khi giữ sào đủ 500 bước
_, cuoi_double = dqn(double=True)                           # Double DQN
with torch.no_grad():                                       # không cần gradient
    for ten, m in (("DQN", ban_cuoi), ("Double DQN", cuoi_double)):   # hai mạng cuối
        print(ten, danh_gia(m, n=100, seed=20_000), round(m(torch.as_tensor(s)).max().item(), 2), round(that, 2))   # điểm test, Q dự đoán, giá trị thật

Output mong đợi:

```text
5000 142 56.6 143.95
10000 211 90.4 100.55
15000 268 85.4 97.5
20000 306 129.4 451.3
25000 317 327.8 259.15
30000 356 146.0 500.0
35000 367 364.1 358.5
40000 423 67.6 88.0
45000 473 91.5 79.45
50000 523 125.2 352.35
DQN 500.0 134.77 99.34
Double DQN 310.53 127.63 99.34
```

### Bài 35.2 · Chép mạng đích thưa hơn

Chạy lại với cap_nhat_dich=500.
<details><summary>Xem đáp án</summary>

Seed 0 không lần nào chạm 500; trên 5 seed chỉ 1 seed chạm 500. Tần suất chép mạng đích phải tương xứng với tổng số bước huấn luyện. Code ở ô ngay dưới.
</details>

In [ ]:
_ = dqn(cap_nhat_dich=500)                                  # chép mạng đích mỗi 500 bước

Output mong đợi:

```text
5000 173 48.4 199.3
10000 210 162.6 151.45
15000 241 170.6 126.7
20000 274 164.4 197.25
25000 309 143.2 136.45
30000 340 137.0 120.35
35000 375 155.0 117.75
40000 411 127.0 110.55
45000 455 114.8 227.65
50000 497 119.1 143.4
```

### Bài 35.3 · Nhầm bị cắt ngang với kết thúc

Chạy lại với ket_thuc_khi_cat=True.
<details><summary>Xem đáp án</summary>

Giống hệt bản đúng tới bước 25 000, rồi tách ra; mạng cuối chỉ được 166,8 trên test. Trên 5 seed, 360,6 so với 479,2. Code ở ô ngay dưới.
</details>

In [ ]:
_, cuoi_sai = dqn(ket_thuc_khi_cat=True)                    # bước 500 cũng bị ghi là kết thúc
with torch.no_grad():                                       # không cần gradient
    print(danh_gia(cuoi_sai, n=100, seed=20_000), round(cuoi_sai(torch.as_tensor(s)).max().item(), 2))   # điểm test, Q ở trạng thái đầu

Output mong đợi:

```text
5000 132 62.0 177.35
10000 193 72.6 95.2
15000 245 98.0 101.2
20000 296 100.7 129.95
25000 349 82.8 90.05
30000 368 251.8 151.25
35000 421 87.2 82.9
40000 457 148.9 500.0
45000 604 21.0 24.9
50000 641 218.5 167.7
166.81 142.33
```